# nb09 — Full Ablation: Loss × Augmentation × Featurizer
> **36-cell grid with heatmaps, Fisher ratio, and silhouette scores.**  
> More thorough than nb02. Resume-safe. Use `SKIP_COMPLETED=true` to spread across sessions.

---

## What this ablation covers

| Axis | Levels | Cells |
|------|--------|-------|
| Featurizer | mfcc, filterbank, sincnet | 3 |
| Loss | bce, focal, rppl, arcface | 4 |
| Augmentation | none, bg_noise, full | 3 |
| **Total** | | **36 cells** |

Each cell takes ~8–15 min on Kaggle T4 → **total ~5–9 h**.
With `SKIP_COMPLETED=true`, you can spread across multiple sessions.

## How to read the heatmap

The main heatmap shows F1 as a function of (featurizer, loss).
- Brighter = higher F1.
- Row differences = featurizer impact.
- Column differences = loss function impact.

The augmentation impact chart shows the lift from `none → bg_noise → full`
separately for each featurizer — useful for deciding whether augmentation
data collection is worth the effort.

## Embedding quality metrics

For the top-5 configurations:
- **Fisher ratio** = inter-class variance / intra-class variance.
  Higher = wake/non-wake embeddings are more separated.
- **Silhouette score** (sklearn) = cohesion vs separation.
  Score in [-1, 1], higher = better cluster quality.

## Expected runtime budget

| Platform | Per cell | Total (36 cells) |
|----------|----------|-------------------|
| Kaggle T4 | ~8–15 min | ~5–9 h |
| Local CPU | ~30–90 min | ~20–50 h (multi-session) |

Reduce `EPOCHS=10` for a faster but noisier ablation.

In [ ]:
import os

# ── Core ──────────────────────────────────────────────────────────────────────
WAKE_WORD         = os.environ.get("WAKE_WORD",         "hey jarvis")
OUTPUT_DIR        = os.environ.get("OUTPUT_DIR",        "./ww_output")
DEVICE            = os.environ.get("DEVICE",            "auto")
SEED              = int(os.environ.get("SEED",          "42"))

# ── Ablation axes ─────────────────────────────────────────────────────────────
FEATURIZERS       = os.environ.get("FEATURIZERS",       "mfcc,filterbank,sincnet").split(",")
LOSSES            = os.environ.get("LOSSES",            "bce,focal,rppl,arcface").split(",")
AUGMENT_LEVELS    = os.environ.get("AUGMENT_LEVELS",    "none,bg_noise,full").split(",")

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS            = int(os.environ.get("EPOCHS",        "20"))
BATCH_SIZE        = int(os.environ.get("BATCH_SIZE",    "16"))
SKIP_COMPLETED    = os.environ.get("SKIP_COMPLETED",    "true").lower() == "true"

# ── Dataset ───────────────────────────────────────────────────────────────────
N_POSITIVE        = int(os.environ.get("N_POSITIVE",    "400"))
LANG              = os.environ.get("LANG",              "en")
ADVERSARIAL       = os.environ.get("ADVERSARIAL",       "true").lower() == "true"
DOWNLOAD_AUGMENT  = os.environ.get("DOWNLOAD_AUGMENT",  "true").lower() == "true"
REUSE_DATASET     = os.environ.get("REUSE_DATASET",     "true").lower() == "true"
CUSTOM_TRAIN_CSV  = os.environ.get("CUSTOM_TRAIN_CSV",  "")
CUSTOM_TEST_CSV   = os.environ.get("CUSTOM_TEST_CSV",   "")

# ── MLflow (optional) ────────────────────────────────────────────────────────
MLFLOW_URI        = os.environ.get("MLFLOW_URI",        "")
MLFLOW_SECRET     = os.environ.get("MLFLOW_SECRET",     "MLFLOW_TOKEN")
MLFLOW_EXPERIMENT = os.environ.get("MLFLOW_EXPERIMENT", "ww_ablation")

N_CELLS = len(FEATURIZERS) * len(LOSSES) * len(AUGMENT_LEVELS)
print(f"Ablation grid: {len(FEATURIZERS)} featurizers × {len(LOSSES)} losses × {len(AUGMENT_LEVELS)} augment = {N_CELLS} cells")
print(f"Featurizers : {FEATURIZERS}")
print(f"Losses      : {LOSSES}")
print(f"Augmentation: {AUGMENT_LEVELS}")
print(f"Epochs/cell : {EPOCHS}")

In [ ]:
import subprocess, sys, os

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("torch", "torchaudio", "soundfile", "numpy", "scikit-learn",
     "matplotlib", "pandas", "seaborn", "librosa", "onnx", "onnxruntime",
     "click", "tqdm")
_pip("ovos-plugin-manager", "ovos-tts-plugin-edge-tts",
     "ovos-vad-plugin-silero", "datasets")
try:
    import ww_trainer
except ImportError:
    _pip("ww_trainer")

_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "paperspace" if os.path.exists("/notebooks")  else
    "colab"      if "google.colab" in sys.modules else
    "local"
)

import torch
torch.set_num_threads(min(12, os.cpu_count() or 4))
os.environ.setdefault("OMP_NUM_THREADS", str(min(12, os.cpu_count() or 4)))

print(f"Platform : {_platform} | CUDA: {torch.cuda.is_available()} | threads: {torch.get_num_threads()}")

In [ ]:
import os

# ── MLflow setup ──────────────────────────────────────────────────────────────
if _platform == "kaggle" and MLFLOW_SECRET:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret(MLFLOW_SECRET)
        os.environ["MLFLOW_TRACKING_TOKEN"] = token
        print(f"MLflow token injected from Kaggle Secret '{MLFLOW_SECRET}'")
    except Exception as e:
        print(f"MLflow secret not found: {e}")

if MLFLOW_URI:
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_URI
    try:
        import mlflow
        mlflow.set_tracking_uri(MLFLOW_URI)
        mlflow.set_experiment(MLFLOW_EXPERIMENT)
        print(f"MLflow experiment: {MLFLOW_EXPERIMENT!r} at {MLFLOW_URI}")
    except Exception as e:
        print(f"MLflow warning: {e}")
else:
    print("MLFLOW_URI not set — tracking disabled")

In [ ]:
import shutil
from pathlib import Path

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
# Ablation needs ~200 MB per cell → 36 cells × 200 MB = ~7 GB
assert free_gb > 5, f"Only {free_gb:.1f} GB free — need at least 5 GB for full ablation."
print(f"Disk free: {free_gb:.1f} GB  (36 cells × ~200 MB ≈ 7 GB needed for full run)")

_aug_kwargs_full = {}

if CUSTOM_TRAIN_CSV:
    import random, csv as _csv
    from ww_trainer.utils import read_dataset_csv
    train_csv = Path(CUSTOM_TRAIN_CSV)
    test_csv  = Path(CUSTOM_TEST_CSV) if CUSTOM_TEST_CSV else None
    if test_csv is None:
        split_dir = Path(OUTPUT_DIR) / "dataset_split"
        split_dir.mkdir(parents=True, exist_ok=True)
        split_train = split_dir / "train.csv"
        split_test  = split_dir / "test.csv"
        if not split_train.exists():
            rows = read_dataset_csv(train_csv)
            random.seed(SEED); random.shuffle(rows)
            cut = int(len(rows) * 0.8)
            for p, rs in [(split_train, rows[:cut]), (split_test, rows[cut:])]:
                with open(p, "w", newline="") as f:
                    _csv.writer(f).writerows(rs)
        train_csv, test_csv = split_train, split_test
else:
    from ww_trainer.datagen import DatagenConfig, run_datagen_pipeline, DatagenResult, normalize_wake_word
    dataset_dir = Path(OUTPUT_DIR) / "dataset"
    _train_csv_check = dataset_dir / "train" / "metadata.csv"
    if REUSE_DATASET and _train_csv_check.exists():
        slug = normalize_wake_word(WAKE_WORD)
        _dr = DatagenResult(
            train_csv=dataset_dir / "train" / "metadata.csv",
            test_csv=dataset_dir / "test" / "metadata.csv",
            positives_dir=dataset_dir / slug / "positives",
            negatives_dir=dataset_dir / slug / "negatives",
            bg_noise_dir=dataset_dir / "augmentation" / "bg_noise",
            music_dir=dataset_dir / "augmentation" / "music",
            rir_dir=dataset_dir / "augmentation" / "rir",
        )
        print(f"Reusing dataset at {dataset_dir}")
    else:
        _dr = run_datagen_pipeline(DatagenConfig(
            wake_word=WAKE_WORD, output_dir=dataset_dir,
            n_positive=N_POSITIVE, lang=LANG,
            adversarial=ADVERSARIAL, vad_trim=True,
            download_augmentation=DOWNLOAD_AUGMENT, seed=SEED,
        ))
    train_csv, test_csv = _dr.train_csv, _dr.test_csv
    for attr, key in [("bg_noise_dir", "bg_noise_folder"),
                      ("music_dir", "music_folder"),
                      ("rir_dir", "rir_folder")]:
        d = getattr(_dr, attr, None)
        if d and Path(d).exists():
            _aug_kwargs_full[key] = str(d)

print(f"train_csv: {train_csv}")
print(f"test_csv : {test_csv}")

In [ ]:
import json, time
from pathlib import Path
from ww_trainer.quickstart import train_from_wakeword

# ── Ablation grid loop (resumable) ────────────────────────────────────────────
# Featurizer maps to tier: mfcc→small, filterbank→filterbank_small, sincnet→sincnet_small

FEAT_TO_TIER = {
    "mfcc":       "small",
    "filterbank": "filterbank_small",
    "sincnet":    "sincnet_small",
    "gammatone":  "gammatone_small",
    "micro":      "micro",
    "delta":      "delta_micro",
}

def _aug_kwargs_for_level(level):
    if level == "none":
        return {}
    if level == "bg_noise":
        return {k: v for k, v in _aug_kwargs_full.items() if "bg_noise" in k}
    return dict(_aug_kwargs_full)

results_dir = Path(OUTPUT_DIR) / "ablation_results"
results_dir.mkdir(parents=True, exist_ok=True)
all_results = []

total = N_CELLS
cell_idx = 0

for feat in FEATURIZERS:
    tier = FEAT_TO_TIER.get(feat, "small")
    for loss in LOSSES:
        for augment in AUGMENT_LEVELS:
            cell_idx += 1
            cell_key  = f"{feat}__{loss}__{augment}"
            result_file = results_dir / f"{cell_key}.json"
            model_subdir = Path(OUTPUT_DIR) / "ablation_models" / cell_key

            print(f"\n[{cell_idx}/{total}] feat={feat!r}  loss={loss!r}  aug={augment!r}")

            if SKIP_COMPLETED and result_file.exists():
                saved = json.loads(result_file.read_text())
                print(f"  SKIP: F1={saved.get('f1', 0):.4f}")
                all_results.append(saved)
                continue

            aug_kw = _aug_kwargs_for_level(augment)
            t0 = time.time()
            try:
                r = train_from_wakeword(
                    WAKE_WORD, str(model_subdir),
                    tier=tier,
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    device=DEVICE,
                    seed=SEED,
                    reuse_dataset=True,
                    losses_cfg=[{"name": loss, "weight": 1.0}],
                    **aug_kw,
                )
                elapsed = time.time() - t0
                row = {
                    "featurizer": feat,
                    "tier": tier,
                    "loss": loss,
                    "augment": augment,
                    "f1": r.metrics.get("f1", 0.0),
                    "precision": r.metrics.get("precision", 0.0),
                    "recall": r.metrics.get("recall", 0.0),
                    "elapsed_s": round(elapsed, 1),
                    "feat_onnx": str(model_subdir / "model" / "best_f1_featurizer.onnx"),
                    "head_onnx": str(r.best_onnx_path) if r.best_onnx_path else "",
                    "status": "ok",
                }
                print(f"  DONE: F1={row['f1']:.4f}  ({elapsed:.0f}s)")
            except Exception as exc:
                elapsed = time.time() - t0
                row = {
                    "featurizer": feat, "tier": tier,
                    "loss": loss, "augment": augment,
                    "f1": 0.0, "precision": 0.0, "recall": 0.0,
                    "elapsed_s": round(elapsed, 1),
                    "feat_onnx": "", "head_onnx": "",
                    "status": f"error: {exc}",
                }
                print(f"  ERROR: {exc}")

            result_file.write_text(json.dumps(row, indent=2))
            all_results.append(row)

print(f"\nGrid complete: {sum(1 for r in all_results if r['status']=='ok')}/{len(all_results)} succeeded.")

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ── Reload all results from disk ──────────────────────────────────────────────
# (handles partial runs and multi-session resumption)

_loaded = []
for f in sorted((Path(OUTPUT_DIR) / "ablation_results").glob("*.json")):
    _loaded.append(json.loads(f.read_text()))

df = pd.DataFrame(_loaded)
df_ok = df[df["status"] == "ok"].copy()

print(f"Loaded {len(df)} cells ({len(df_ok)} successful)")

if df_ok.empty:
    print("No successful cells yet — run Cell 6 first.")
else:
    # ── Pivot table: featurizer × loss (mean F1 over augment levels) ──────
    pivot = df_ok.pivot_table(
        index="featurizer", columns="loss", values="f1", aggfunc="mean"
    )
    print("\nF1 pivot (mean over augmentation levels):")
    print(pivot.round(4).to_string())

    # ── Heatmap ───────────────────────────────────────────────────────────
    try:
        import seaborn as sns
        fig, ax = plt.subplots(figsize=(max(6, len(LOSSES) * 1.5),
                                        max(3, len(FEATURIZERS) * 1.2)))
        sns.heatmap(
            pivot, annot=True, fmt=".3f", cmap="YlOrRd",
            vmin=0, vmax=1, ax=ax,
            linewidths=0.5, linecolor="white",
        )
        ax.set_title(f"Ablation heatmap: featurizer × loss (mean F1)\n{WAKE_WORD!r}")
        ax.set_xlabel("Loss function")
        ax.set_ylabel("Featurizer")
        plt.tight_layout()
        heatmap_path = str(Path(OUTPUT_DIR) / "ablation_heatmap.png")
        plt.savefig(heatmap_path, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Heatmap saved: {heatmap_path}")
    except ImportError:
        print("seaborn not installed — skipping heatmap. Run: pip install seaborn")
        print(pivot.round(4).to_string())

    # Top 5
    top5 = df_ok.nlargest(5, "f1")[["featurizer","loss","augment","f1","precision","recall"]]
    print("\nTop 5 configurations:")
    print(top5.to_string(index=False))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# ── Augmentation impact bar chart ─────────────────────────────────────────────
# For each featurizer, show mean F1 at each augmentation level (averaged over losses).
# This isolates the effect of augmentation from the effect of loss/featurizer.

if df_ok.empty:
    print("No data yet.")
else:
    aug_pivot = df_ok.pivot_table(
        index="featurizer", columns="augment", values="f1", aggfunc="mean"
    )

    # Ensure augment columns are in the right order
    _aug_order = [a for a in ["none", "bg_noise", "full"] if a in aug_pivot.columns]
    aug_pivot = aug_pivot[_aug_order]

    print("Augmentation impact (mean F1 over losses):")
    print(aug_pivot.round(4).to_string())

    n_feats = len(aug_pivot)
    n_augs  = len(_aug_order)
    x = np.arange(n_feats)
    width = 0.25
    colors = ["#d9d9d9", "#6baed6", "#2171b5"]

    fig, ax = plt.subplots(figsize=(max(7, n_feats * 2), 4))
    for i, (aug_level, color) in enumerate(zip(_aug_order, colors)):
        if aug_level in aug_pivot.columns:
            vals = aug_pivot[aug_level].values
            bars = ax.bar(x + (i - n_augs/2 + 0.5) * width, vals,
                          width, label=aug_level, color=color, edgecolor="white")

    ax.set_xticks(x)
    ax.set_xticklabels(aug_pivot.index, rotation=15)
    ax.set_ylabel("F1 (mean over losses)")
    ax.set_title(f"Augmentation impact by featurizer — {WAKE_WORD!r}")
    ax.set_ylim(0, 1.05)
    ax.legend(title="Augmentation", fontsize=9)

    plt.tight_layout()
    aug_path = str(Path(OUTPUT_DIR) / "augmentation_impact.png")
    plt.savefig(aug_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Augmentation impact chart saved: {aug_path}")

    # Delta: none → full
    if "none" in aug_pivot.columns and "full" in aug_pivot.columns:
        delta = (aug_pivot["full"] - aug_pivot["none"]).rename("f1_lift_none_to_full")
        print("\nF1 lift (none → full augmentation):")
        print(delta.round(4).to_string())

In [ ]:
import numpy as np
import csv
import torchaudio
import onnxruntime as ort
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import silhouette_score

# ── Embedding quality: Fisher ratio + silhouette for top-5 configs ─────────────
# Fisher ratio = inter-class variance / intra-class variance.
# Silhouette score measures cluster cohesion vs separation in [-1, 1].
# Both metrics are computed on featurizer embeddings (not classifier outputs).

if df_ok.empty:
    print("No successful runs yet.")
else:
    # Load test samples once
    _wavs_eq, _labels_eq = [], []
    with open(test_csv) as f:
        for row in csv.reader(f):
            if len(row) < 2 or not Path(row[0]).exists():
                continue
            wav, sr = torchaudio.load(row[0])
            if sr != 16000:
                wav = torchaudio.functional.resample(wav, sr, 16000)
            _wavs_eq.append(wav.mean(0).numpy().astype(np.float32))
            _labels_eq.append(int(row[1].strip()))
            if len(_wavs_eq) >= 200:
                break
    _labels_arr = np.array(_labels_eq)
    print(f"Test samples: {len(_wavs_eq)} ({(_labels_arr==1).sum()} pos, {(_labels_arr==0).sum()} neg)")

    def _extract_embs(feat_onnx_path, wavs):
        sess = ort.InferenceSession(str(feat_onnx_path), providers=["CPUExecutionProvider"])
        in_name = sess.get_inputs()[0].name
        embs = []
        for wav in wavs:
            out = sess.run(None, {in_name: wav[np.newaxis, :]})[0]
            embs.append(out.mean(axis=1).ravel() if out.ndim == 3 else out.ravel())
        return np.array(embs)

    def _fisher_ratio(embs, labels):
        """Fisher ratio: inter-class / intra-class variance (mean over dims)."""
        pos = embs[labels == 1]
        neg = embs[labels == 0]
        if len(pos) < 2 or len(neg) < 2:
            return float("nan")
        mu_pos, mu_neg = pos.mean(0), neg.mean(0)
        mu_all = embs.mean(0)
        sb = 0.5 * (np.sum((mu_pos - mu_all)**2) + np.sum((mu_neg - mu_all)**2))
        sw = 0.5 * (pos.var(0).sum() + neg.var(0).sum())
        return float(sb / (sw + 1e-8))

    top5 = df_ok.nlargest(5, "f1")
    eq_rows = []
    for _, row in top5.iterrows():
        feat_p = Path(row["feat_onnx"]) if row.get("feat_onnx") else None
        if feat_p is None or not feat_p.exists():
            print(f"  {row['featurizer']}/{row['loss']}/{row['augment']}: feat ONNX missing")
            continue
        try:
            embs = _extract_embs(feat_p, _wavs_eq)
            fisher = _fisher_ratio(embs, _labels_arr)
            sil = silhouette_score(embs, _labels_arr) if len(np.unique(_labels_arr)) > 1 else float("nan")
            eq_rows.append({
                "featurizer": row["featurizer"],
                "loss": row["loss"],
                "augment": row["augment"],
                "f1": row["f1"],
                "fisher_ratio": round(fisher, 4),
                "silhouette": round(sil, 4),
            })
            print(f"  {row['featurizer']:12s} {row['loss']:8s} {row['augment']:10s}  "
                  f"F1={row['f1']:.3f}  Fisher={fisher:.3f}  Sil={sil:.3f}")
        except Exception as e:
            print(f"  Error for {row['featurizer']}/{row['loss']}: {e}")

    if eq_rows:
        df_eq = pd.DataFrame(eq_rows)
        print()
        print("Embedding quality summary:")
        print(df_eq[["featurizer","loss","augment","f1","fisher_ratio","silhouette"]].to_string(index=False))

        # Bar chart: F1 vs Fisher vs Silhouette for top-5
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        labels_plot = [
            f"{r['featurizer']}\n{r['loss']}\n{r['augment']}" for r in eq_rows
        ]
        for ax, metric, color, title in [
            (axes[0], "f1", "steelblue", "F1"),
            (axes[1], "fisher_ratio", "coral", "Fisher ratio (higher=better)"),
            (axes[2], "silhouette", "seagreen", "Silhouette score (higher=better)"),
        ]:
            vals = [r[metric] for r in eq_rows]
            ax.bar(range(len(eq_rows)), vals, color=color, edgecolor="white")
            ax.set_xticks(range(len(eq_rows)))
            ax.set_xticklabels(labels_plot, fontsize=7)
            ax.set_title(title)
            ax.set_ylabel(metric)

        plt.suptitle(f"Embedding quality — top-5 configs — {WAKE_WORD!r}", fontsize=11)
        plt.tight_layout()
        eq_path = str(Path(OUTPUT_DIR) / "embedding_quality.png")
        plt.savefig(eq_path, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"Embedding quality chart saved: {eq_path}")

In [ ]:
import json
import csv
import numpy as np
import torchaudio
from pathlib import Path
from ww_trainer.inference import OnnxWakeWordInferencer

# ── Best config summary + ONNX verification ───────────────────────────────────

if df_ok.empty:
    print("No successful cells yet.")
else:
    best = df_ok.nlargest(1, "f1").iloc[0]

    print("=" * 60)
    print(f"Best ablation configuration — {WAKE_WORD!r}")
    print(f"  Featurizer : {best['featurizer']}")
    print(f"  Loss       : {best['loss']}")
    print(f"  Augment    : {best['augment']}")
    print(f"  F1         : {best['f1']:.4f}")
    print(f"  Precision  : {best['precision']:.4f}")
    print(f"  Recall     : {best['recall']:.4f}")
    print()

    # ONNX verification
    feat_p = Path(best["feat_onnx"]) if best.get("feat_onnx") else None
    head_p = Path(best["head_onnx"]) if best.get("head_onnx") else None

    for label, p in [("featurizer", feat_p), ("head", head_p)]:
        if p and p.exists():
            print(f"  OK  {label}: {p.name}  ({p.stat().st_size/1024:.0f} KB)")
        else:
            print(f"  MISSING  {label}: {p}")

    # Inference test
    if feat_p and feat_p.exists() and head_p and head_p.exists():
        inferencer = OnnxWakeWordInferencer(str(feat_p), str(head_p))
        _pos_path = None
        with open(test_csv) as f:
            for row in csv.reader(f):
                if len(row) >= 2 and row[1].strip() == "1" and Path(row[0]).exists():
                    _pos_path = row[0]; break
        if _pos_path:
            wav, sr = torchaudio.load(_pos_path)
            if sr != 16000:
                wav = torchaudio.functional.resample(wav, sr, 16000)
            score = inferencer.infer(wav.mean(0).numpy().astype(np.float32))
            print(f"\nInference test: score={score:.4f}  ({'PASS' if score > 0.5 else 'LOW'})")

    print()
    print("CLI commands:")
    if feat_p and feat_p.exists() and head_p and head_p.exists():
        print(f"  .venv/bin/python scripts/eval/test_wakeword.py \\")
        print(f"      --featurizer {feat_p} \\")
        print(f"      --model      {head_p} \\")
        print(f"      --audio      sample.wav")
    print()
    print("Output files:")
    for f in [
        Path(OUTPUT_DIR) / "ablation_heatmap.png",
        Path(OUTPUT_DIR) / "augmentation_impact.png",
        Path(OUTPUT_DIR) / "embedding_quality.png",
    ]:
        if f.exists():
            print(f"  {f}")
    print("=" * 60)